In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)  # Display full content of each column
pd.set_option('display.max_columns', None)   # Display all columns
pd.set_option('display.width', 5000)         # Set display width

REGEX
--

In [2]:
import re
import pandas as pd

# Rule-based extractor: the original single regex only covered the
# "MODE/DR|CR/ID/NAME/BANK/UPIID" slash-delimited transfer format. Real SBI
# statements contain several other narration shapes (plain IMPS without an
# explicit DR/CR marker, star-delimited NEFT, ATM withdrawals, POS purchases,
# bank-generated interest/fee/insurance lines, UPI reversals, generic INB
# entity payments). Each shape gets its own small regex + handler, tried in
# priority order; the original weak split-based parser remains as the final
# fallback for anything still unmatched.

DRCR_BY_TYPE = {"WDL TFR": "DR", "DEP TFR": "CR"}


def _type_of(m):
    t = m.group("type").upper()
    return "WDL TFR" if t.startswith("WDL") else "DEP TFR"


def _base(m, **overrides):
    t = _type_of(m)
    out = {
        "Transaction_Type": t,
        "Transaction_Mode": "N/A",
        "DR/CR_Indicator": DRCR_BY_TYPE[t],
        "Transaction_ID": "N/A",
        "Recipient_Name": "N/A",
        "Bank": "N/A",
        "UPI_ID": "N/A",
        "Note": "N/A",
    }
    out.update(overrides)
    return pd.Series(out)


# --- Rule 1: primary slash-delimited transfer (UPI/INB/IMPS/NEFT etc with explicit DR|CR marker) ---
# mode/bank are case-insensitive: some UPI handles record the bank code in
# lowercase (e.g. "utib"), which silently fell through to the weak fallback before.
RULE_MAIN = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+"
    r"(?P<mode>[A-Za-z]+)/(?P<drcr>DR|CR)/"
    r"(?P<id>\d+)/"
    r"(?P<name>[^/]+)/"
    r"(?P<bank>[A-Za-z]+)/"
    r"(?P<upi_id>[^\s/]+)"
    r"(?:\s*-?\d+/(?P<note>[A-Za-z]+))?"
)


def handle_main(m):
    return pd.Series({
        "Transaction_Type": _type_of(m),
        "Transaction_Mode": m.group("mode").upper(),
        "DR/CR_Indicator": m.group("drcr").upper(),
        "Transaction_ID": m.group("id"),
        "Recipient_Name": m.group("name").strip(),
        "Bank": m.group("bank").upper(),
        "UPI_ID": m.group("upi_id"),
        "Note": m.group("note") if m.group("note") else "N/A",
    })


# --- Rule 2: IMPS variant A — INB IMPS<id>/<sender>/XX<acct>/<name>   <ref> AT ---
RULE_IMPS_A = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+INB\s+IMPS(?P<id>\d+)/(?P<sender>[^/]+)/XX(?P<acct>\d+)/\s*"
    r"(?P<name>[^\d/]+?)\s+\d{6,}\s+AT"
)


def handle_imps_a(m):
    name = m.group("name").strip()
    return _base(m, Transaction_Mode="IMPS", Transaction_ID=m.group("id"),
                 Recipient_Name=name if name else "N/A")


# --- Rule 3: IMPS variant B — (INB )?IMPS/<id>/<bank>-XX<acct>-<sender> /<nickname>   <ref> AT ---
# "INB" prefix and the "XX" account-suffix marker are both sometimes lowercase.
RULE_IMPS_B = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+(?:INB\s+)?IMPS/(?P<id>\d+)/(?P<bank>[A-Za-z0-9]{2,5})-[Xx]{2}(?P<acct>\d+)-"
    r"(?P<sender>[^/]+?)\s*/\s*(?P<nickname>[^\d/]*?)\s+\d{6,}\s+AT"
)


def handle_imps_b(m):
    sender = m.group("sender").strip()
    remark = m.group("nickname").strip()
    # In this narration shape the counterparty name sits in the <sender>
    # segment (bank-XXacct-<name>); the trailing field is a free-text
    # remark/purpose (e.g. "Loan Rep", "Bill Pay", "IMPS tran"), NOT a payee.
    # A wrapped statement export can also mangle that remark (e.g. "Loan Rep"
    # split across a line becomes "Loa n Rep"), which is exactly why it must
    # not be used as the recipient. Prefer the counterparty name; fall back to
    # the remark only when the name segment is empty.
    name = sender if sender else remark
    note = remark if remark and remark.upper() != "NA" else "N/A"
    return _base(m, Transaction_Mode="IMPS", Transaction_ID=m.group("id"),
                 Bank=m.group("bank").upper(), Recipient_Name=name if name else "N/A",
                 Note=note)


# --- Rule 4: NEFT star-delimited — NEFT*<IFSC>*<UTR>*<NAME>   <ref> AT ---
RULE_NEFT = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+NEFT\*(?P<ifsc>[A-Za-z0-9]+)\*(?P<utr>[A-Za-z0-9]+)\*"
    r"(?P<name>[^\d/]+?)\s+\d{6,}\s+AT"
)


def handle_neft(m):
    ifsc = m.group("ifsc")
    bank = ifsc[:4].upper() if len(ifsc) >= 4 else "N/A"
    return _base(m, Transaction_Mode="NEFT", Transaction_ID=m.group("utr"),
                 Bank=bank, Recipient_Name=m.group("name").strip())


# --- Rule 5: UPI reversal — either "UPI/<id>/REVERSAL" or "UPI/REV/<id>" ---
RULE_UPI_REVERSAL = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+UPI/(?:(?P<id1>\d+)/REVERSAL|REV/(?P<id2>\d+))"
)


def handle_upi_reversal(m):
    txn_id = m.group("id1") or m.group("id2") or "N/A"
    return _base(m, Transaction_Mode="UPI", Transaction_ID=txn_id, Recipient_Name="REVERSAL")


# --- Rule 6: ATM withdrawal ---
# Leading atm-code digits (sometimes with a stray embedded space) and an optional
# lone "+" marker before the location are stripped; whatever remains is the location.
RULE_ATM = re.compile(r"ATM\s*WDL\s+ATM\s*CASH\s+(?P<rest>.+)", re.IGNORECASE)


def handle_atm(m):
    rest = m.group("rest")
    cleaned = re.sub(r"^[\d\s]+", "", rest)
    cleaned = re.sub(r"^\+\s*", "", cleaned)
    location = re.sub(r"\s+", " ", cleaned).strip()
    return pd.Series({
        "Transaction_Type": "WDL TFR", "Transaction_Mode": "ATM", "DR/CR_Indicator": "DR",
        "Transaction_ID": "N/A", "Recipient_Name": "ATM_WITHDRAWAL",
        "Bank": "N/A", "UPI_ID": "N/A", "Note": location or "N/A",
    })


# --- Rule 7: ATM card fee/AMC ---
RULE_ATM_FEE = re.compile(r"DEBIT\s+ATMCard\s+AMC", re.IGNORECASE)


def handle_atm_fee(_m):
    return pd.Series({
        "Transaction_Type": "WDL TFR", "Transaction_Mode": "FEE", "DR/CR_Indicator": "DR",
        "Transaction_ID": "N/A", "Recipient_Name": "BANK_FEE",
        "Bank": "N/A", "UPI_ID": "N/A", "Note": "N/A",
    })


# --- Rule 8: interest credit (narration sometimes has a stray space: "INTERES T CREDIT") ---
RULE_INTEREST = re.compile(r"INTERES\s*T?\s*CREDIT", re.IGNORECASE)


def handle_interest(_m):
    return pd.Series({
        "Transaction_Type": "DEP TFR", "Transaction_Mode": "INTEREST", "DR/CR_Indicator": "CR",
        "Transaction_ID": "N/A", "Recipient_Name": "INTEREST_CREDIT",
        "Bank": "N/A", "UPI_ID": "N/A", "Note": "N/A",
    })


# --- Rule 9: insurance premium (PMJJBY / PMSBY govt schemes) ---
RULE_INSURANCE = re.compile(r"PMJJBY|PMSBY", re.IGNORECASE)


def handle_insurance(_m):
    return pd.Series({
        "Transaction_Type": "WDL TFR", "Transaction_Mode": "INSURANCE", "DR/CR_Indicator": "DR",
        "Transaction_ID": "N/A", "Recipient_Name": "INSURANCE_PREMIUM",
        "Bank": "N/A", "UPI_ID": "N/A", "Note": "N/A",
    })


# --- Rule 10: cheque clearing ---
RULE_CHEQUE = re.compile(r"CLEARING\s*/\s*CHEQUE", re.IGNORECASE)


def handle_cheque(_m):
    return pd.Series({
        "Transaction_Type": "N/A", "Transaction_Mode": "CHEQUE", "DR/CR_Indicator": "N/A",
        "Transaction_ID": "N/A", "Recipient_Name": "CHEQUE_CLEARING",
        "Bank": "N/A", "UPI_ID": "N/A", "Note": "N/A",
    })


# --- Rule 11: POS card purchase (merchant name/city run together with no delimiter — best effort) ---
RULE_POS = re.compile(r"POS\s+ATM\s+PURCH.*?\d{4,}\s*(?P<name>[A-Z][A-Za-z ]+)$")


def handle_pos(m):
    name = re.sub(r"\s+", " ", m.group("name")).strip()
    return pd.Series({
        "Transaction_Type": "WDL TFR", "Transaction_Mode": "POS", "DR/CR_Indicator": "DR",
        "Transaction_ID": "N/A", "Recipient_Name": name if name else "N/A",
        "Bank": "N/A", "UPI_ID": "N/A", "Note": "N/A",
    })


# --- Rule 12: generic "OF <name> AT <branch>" catch-all (SBILT internal transfers, tax payments, etc.) ---
RULE_OF_NAME = re.compile(r"(?P<type>(?:WDL|DEP)\s+TFR).*?\bO\s?F\b\s+(?P<name>.+?)\s+AT\s+\d")


def handle_of_name(m):
    name = re.sub(r"\s+", " ", m.group("name")).strip()
    return _base(m, Transaction_Mode="INB", Recipient_Name=name if name else "N/A")


# --- Rule 13: generic INB <entity>   <ref> AT (no OF marker, no slash fields) ---
RULE_INB_ENTITY = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+INB\s+(?P<name>[A-Za-z][A-Za-z0-9 ._-]{3,}?)\s+\d{6,}\s*(?:OF|AT)"
)


def handle_inb_entity(m):
    name = re.sub(r"\s+", " ", m.group("name")).strip()
    return _base(m, Transaction_Mode="INB", Recipient_Name=name if name else "N/A")


RULES = [
    (RULE_MAIN, handle_main),
    (RULE_IMPS_B, handle_imps_b),
    (RULE_IMPS_A, handle_imps_a),
    (RULE_NEFT, handle_neft),
    (RULE_UPI_REVERSAL, handle_upi_reversal),
    (RULE_ATM, handle_atm),
    (RULE_ATM_FEE, handle_atm_fee),
    (RULE_INTEREST, handle_interest),
    (RULE_INSURANCE, handle_insurance),
    (RULE_CHEQUE, handle_cheque),
    (RULE_POS, handle_pos),
    (RULE_OF_NAME, handle_of_name),
    (RULE_INB_ENTITY, handle_inb_entity),
]

# List of all transaction modes, used only by the legacy fallback below
transaction_modes = [
    "UPI", "INB", "IMP", "NEFT", "RTGS", "Cheque", "Cash Deposit", "Cash Withdrawal",
    "POS", "DD", "SWIFT", "Wire Transfer", "ECS", "Bill Pay", "M-wallet", "EMI", "EFT", "ACH"
]


def fallback_extract(description_clean):
    """Legacy weak parser, last resort for anything no rule above matches.
    Fixed vs. the original: WDL/DR is now assigned symmetrically with DEP/CR
    (previously only DEP TFR / CR ever got set here)."""
    split_data = description_clean.split('/')

    transaction_mode = "N/A"
    for mode in transaction_modes:
        if mode in description_clean:
            transaction_mode = mode
            break

    note_match = re.search(r'/\s*([A-Za-z]+)\s+\d+', description_clean)
    note = note_match.group(1) if note_match else "N/A"

    txn_id_match = re.search(r'\d{6,}', split_data[0])
    txn_id = txn_id_match.group() if txn_id_match else "N/A"

    is_dep = "DEP TFR" in description_clean
    is_wdl = "WDL TFR" in description_clean

    return pd.Series({
        "Transaction_Type": "DEP TFR" if is_dep else ("WDL TFR" if is_wdl else "N/A"),
        "Transaction_Mode": transaction_mode,
        "DR/CR_Indicator": "CR" if is_dep else ("DR" if is_wdl else "N/A"),
        "Transaction_ID": txn_id,
        "Recipient_Name": split_data[1] if len(split_data) > 1 else "N/A",
        "Bank": "N/A",
        "UPI_ID": "N/A",
        "Note": note
    })


def extract_details(description):
    if pd.isna(description):
        return pd.Series({
            "Transaction_Type": "N/A",
            "Transaction_Mode": "N/A",
            "DR/CR_Indicator": "N/A",
            "Transaction_ID": "N/A",
            "Recipient_Name": "N/A",
            "Bank": "N/A",
            "UPI_ID": "N/A",
            "Note": "N/A"
        })

    description_clean = str(description).replace("\n", " ").strip()

    for pattern, handler in RULES:
        match = pattern.search(description_clean)
        if match:
            return handler(match)

    return fallback_extract(description_clean)
# text = """ DEP TFR   UPI/CR/345946067401/SNEHA DI/HDFC/ss280387
#  -2/UPI   0093009162091 AT 71097 EVERSHINE CITY BRANCH"""
# print(extract_details(text))

Rename Columns
---

In [ ]:
# Read the RAW workbook produced by CSV_PARSER.ipynb. Read it straight from the
# path CSV_PARSER writes ("CSVS/SBI/...") rather than a hand-copied root-level
# duplicate: an earlier copy at "CSVS/SpendWise_4yrs_RAW.xlsx" had its dates
# corrupted (Excel serials mis-read as ms-since-epoch -> every date collapsed to
# 1970-01-01), so reading the canonical source avoids that stale, broken copy.
df = pd.read_excel("CSVS/SBI/SpendWise_4yrs_RAW.xlsx")

df = df.replace('\n', '', regex=True)
# Collapse repeated whitespace so near-identical narration text (e.g. a stray
# extra space from the export) doesn't slip past the exact-match dedup below.
df['Details'] = df['Details'].str.replace(r'\s+', ' ', regex=True).str.strip()

# The yearly export files have overlapping date ranges (e.g. 2026.xlsx and
# 2027.xlsx both cover part of Jan-Mar 2026), so the same ledger line can
# appear twice after merging. Date + Details + Balance together uniquely
# identify a single real transaction line.
rows_before_dedup = len(df)
is_duplicate = df.duplicated(subset=['Date', 'Details', 'Balance'], keep='first')
removed_rows = df[is_duplicate]
if len(removed_rows) > 0:
    print(f"Removing {len(removed_rows)} duplicate rows:\n")
    print(removed_rows[['Date', 'Details', 'Debit', 'Credit', 'Balance']].to_string())
    print()
df = df[~is_duplicate].reset_index(drop=True)
print(f"Dropped {rows_before_dedup - len(df)} duplicate rows from overlapping yearly exports "
      f"({rows_before_dedup} -> {len(df)})")

# Per-file footer rows are already stripped in the merge step above; this is
# just a safety net for any stray fully-blank row.
blank_rows = df.apply(
    lambda row: all(pd.isna(x) or str(x).strip() == "" for x in row),
    axis=1
)
df = df.loc[~blank_rows].reset_index(drop=True)

df = df.rename(columns={
    'Date': 'Transaction_Date',    
    'Details': 'Description',
    'Ref No./Cheque\nNo.': 'Reference No./Cheque No.',
    'Debit': 'Debit',
    'Credit': 'Credit',
    'Balance': 'Balance'
})
df_extracted = df['Description'].apply(extract_details)

df = pd.concat([df, df_extracted], axis=1)
df


Drop Columns
---

In [4]:
df = df.drop(columns=['Ref No/Cheque No', 'Transaction_Type', 'Description'])

Formatting
---

In [5]:
# Convert to datetime
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='mixed', dayfirst=True)
df['Transaction_Date'] = df['Transaction_Date'].dt.date.astype(str)

# Numeric cleaning
df['Debit'] = pd.to_numeric(df['Debit'].replace({',': ''}, regex=True), errors='coerce')
df['Credit'] = pd.to_numeric(df['Credit'].replace({',': ''}, regex=True), errors='coerce')
df['Balance'] = pd.to_numeric(df['Balance'].replace({',': ''}, regex=True), errors='coerce')

# Fill debit/credit
df['Debit'] = df['Debit'].fillna(0)
df['Credit'] = df['Credit'].fillna(0)

# DR/CR logic
def determine_dr_cr(df):
    if df.loc[0, 'Credit'] > 0:
        df.loc[0, 'DR/CR_Indicator'] = 'CR'
    elif df.loc[0, 'Debit'] > 0:
        df.loc[0, 'DR/CR_Indicator'] = 'DR'
    else:
        df.loc[0, 'DR/CR_Indicator'] = None
    
    for i in range(1, len(df)):
        balance_diff = df.loc[i, 'Balance'] - df.loc[i-1, 'Balance']
        df.loc[i, 'DR/CR_Indicator'] = 'CR' if balance_diff > 0 else 'DR'
    
    return df

df = determine_dr_cr(df)

# Amount
df['Amount'] = df['Credit'] - df['Debit']

In [6]:
print(df.isna().sum())


Transaction_Date    0
Debit               0
Credit              0
Balance             0
Transaction_Mode    0
DR/CR_Indicator     0
Transaction_ID      0
Recipient_Name      0
Bank                0
UPI_ID              0
Note                0
Amount              0
dtype: int64


In [7]:
for i in df.columns:
    print(i)
# df

Transaction_Date
Debit
Credit
Balance
Transaction_Mode
DR/CR_Indicator
Transaction_ID
Recipient_Name
Bank
UPI_ID
Note
Amount


In [8]:
import pandas as pd
import numpy as np
import math

def clean_recipient(name):
    if pd.isna(name):
        return "UNKNOWN"
    name = str(name).strip()
    if name.isdigit():
        return "PHONE_TRANSFER"
    return name

df["Recipient_Name"] = df["Recipient_Name"].apply(clean_recipient)

df.replace("N/A", None, inplace=True)
df = df.replace([np.inf, -np.inf], None)
df = df.astype(object)
df = df.where(pd.notna(df), None)

records = df.to_dict(orient="records")

def clean_record(r):
    clean = {}
    for k, v in r.items():
        if v is None:
            clean[k] = None
        elif isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
            clean[k] = None
        elif str(v).lower() == 'nan':   # 🔥 IMPORTANT FIX
            clean[k] = None
        else:
            clean[k] = v
    return clean

records = [clean_record(r) for r in records]

for r in records:
    for v in r.values():
        if isinstance(v, float) and math.isnan(v):
            print("❌ FLOAT NaN FOUND:", r)
        if str(v).lower() == 'nan':
            print("❌ STRING NaN FOUND:", r)

print("✅ 100% CLEAN DATA - READY FOR INSERT")

✅ 100% CLEAN DATA - READY FOR INSERT


Export
---

In [9]:
df.to_excel("CSVS\\SpendWise_4yrs_Clean.xlsx", index=False)

In [10]:
import math
def validate_dataframe(df):
    errors = []

    for idx, row in df.iterrows():
        row_errors = []

        for col, val in row.items():

            # Check None (will become null in JSON)
            if val is None:
                continue  # allowed

            # Check invalid floats (extra safety)
            if isinstance(val, float):
                if math.isnan(val) or math.isinf(val):
                    row_errors.append(f"{col} = Invalid float ({val})")

            # Check empty strings
            if isinstance(val, str) and val.strip() == "":
                row_errors.append(f"{col} = Empty string")

        if row_errors:
            errors.append({
                "row_index": idx,
                "errors": row_errors,
                "row_data": row.to_dict()
            })

    return errors

# -----------------------------
# Run Validation
# -----------------------------
errors = validate_dataframe(df)

# -----------------------------
# Result
# -----------------------------
if not errors:
    print("✅ Data is CLEAN and validated!")

    records = df.to_dict(orient="records")
    print(f"Total records ready: {len(records)}")

else:
    print(f"❌ Found {len(errors)} problematic rows\n")

    for e in errors[:5]:  # show first 5 errors
        print(f"Row {e['row_index']}:")
        for err in e["errors"]:
            print("  -", err)
        print()

    # Save bad rows for debugging
    bad_rows = [e["row_data"] for e in errors]
    pd.DataFrame(bad_rows).to_excel("bad_data.xlsx", index=False)

    print("⚠️ Bad rows exported to 'bad_data.xlsx'")

✅ Data is CLEAN and validated!
Total records ready: 1917


In [11]:
print(df.isna().sum())

Transaction_Date      0
Debit                 0
Credit                0
Balance               0
Transaction_Mode      0
DR/CR_Indicator       0
Transaction_ID       53
Recipient_Name        0
Bank                110
UPI_ID              198
Note                993
Amount                0
dtype: int64


In [12]:
na_count_per_column = df.applymap(lambda x: x == "N/A").sum()

# Print the result
print("Number of 'N/A' values per column:")
print(na_count_per_column)

Number of 'N/A' values per column:
Transaction_Date    0
Debit               0
Credit              0
Balance             0
Transaction_Mode    0
DR/CR_Indicator     0
Transaction_ID      0
Recipient_Name      0
Bank                0
UPI_ID              0
Note                0
Amount              0
dtype: int64


In [13]:
transaction_modes = [
    "UPI",               # Unified Payments Interface
    "INB",               # Internet Banking
    "IMP",               # Immediate Payment Service
    "NEFT",              # National Electronic Funds Transfer
    "RTGS",              # Real-Time Gross Settlement
    "Cheque",            # Paper Cheque
    "Cash Deposit",      # Cash Deposit to account
    "Cash Withdrawal",   # Withdrawal using cash
    "POS",               # Point of Sale (Card Payment)
    "DD",                # Demand Draft
    "SWIFT",             # Society for Worldwide Interbank Financial Telecommunication
    "Wire Transfer",     # Electronic bank transfer
    "ECS",               # Electronic Clearing Service
    "Bill Pay",          # Utility Bill Payments
    "M-wallet",          # Mobile Wallet Transfer
    "EMI",               # Equated Monthly Installment
    "EFT",               # Electronic Funds Transfer
    "ACH",               # Automated Clearing House
]
